<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_04_mlp_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_01 -  SEQ2ONE TUNING - Modelo MLP - Target: `delta_60`**

En esta sección iniciamos el proceso de **tuning del modelo MLP bajo el enfoque many-to-one (SEQ2ONE)**, utilizando como variable objetivo el **`delta_60`**, es decir, la variación en puntos del MNQ en los próximos 60 minutos.

El enfoque SEQ2ONE consiste en utilizar una ventana histórica de tamaño \( L \) minutos como entrada y predecir un único valor futuro correspondiente al horizonte seleccionado. En este caso:

$$
X \in \mathbb{R}^{(L \times F)} \quad \longrightarrow \quad y \in \mathbb{R}
$$

donde:

- $L$ = window size  
- $F$ = número de features (36)
- $y$ = `delta_60`  

---

**Motivación**

De acuerdo con el workflow de investigación y modelado presentado en el libro *Machine Learning for Algorithmic Trading*, el diseño y tuning del modelo corresponde a la etapa de:

> **Design, tune, and evaluate ML models to generate trading signals**

En esta fase, el objetivo no es todavía el backtesting completo, sino encontrar una configuración que:

- Generalice correctamente (early stopping sobre VALID)  
- Maximice capacidad predictiva (R², RMSE, MAE)  
- Mantenga estabilidad direccional (DA)  

---

**Objetivo del tuning**

El propósito del tuning será:

1. Evaluar distintos tamaños de ventana $L$.  
2. Ajustar hiperparámetros del MLP:
   - `hidden_dim`
   - `dropout`
   - `learning_rate`
   - `weight_decay`
3. Seleccionar la mejor configuración usando exclusivamente el conjunto **VALID**.  
4. Reservar el conjunto **TEST** para evaluación final no sesgada.  

Este procedimiento evita leakage y respeta el principio de generalización fuera de muestra, fundamental en modelado financiero.

---

**Contexto técnico**

El MLP:

- Recibe ventanas aplanadas en formato 2D.  
- No modela memoria temporal explícita (a diferencia de LSTM/GRU).  
- Aprende relaciones no lineales entre patrones recientes del mercado y el `delta_60`.  

Por lo tanto, el tamaño de ventana seleccionado determinará indirectamente cuánta información temporal puede capturar el modelo.

---

En las siguientes secciones se definirá el espacio de búsqueda y se ejecutará el proceso de tuning controlado sobre TRAIN/VALID.



# **BLOQUE DE EJECUCIÓN COMPLETO**

## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [3]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

window_sizes = [30, 60, 90, 120, 180]
targets = ['delta_60', 'delta_90', 'ret_60', 'ret_90']
splits = ['train', 'valid', 'test']

In [4]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [5]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl'),
 'delta_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_90.pkl'),
 'ret_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_60.pkl'),
 'ret_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_90.pkl')}

## **4. Reproducibilidad**

In [6]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [7]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [8]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **6. Carga de data windows**

In [9]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y


In [10]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [11]:
from typing import Any, Dict, Mapping
from pathlib import Path

# --------------------------------------------------
# Carga completa: ventanas + scaler por window_size y target
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path
    scalers_path[target] -> Path
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]
    scaler_path = scalers_paths[target]

    # --------------------------
    # 3) Carga
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    scaler = load_scaler(scaler_path)

    # --------------------------
    # 4) Inferir horizonte
    # --------------------------
    horizon = int(target.split("_")[-1])

    # --------------------------
    # 5) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [12]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [13]:
def create_bundles(window_size, targets: list, windows_paths=windows_paths, scalers_paths=scalers_paths, *, flatten_X=False):

    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
        )
        if flatten_X:
            b["train"]["X"] = maybe_flatten_X(b["train"]["X"], flatten=True)
            b["valid"]["X"] = maybe_flatten_X(b["valid"]["X"], flatten=True)
            b["test"]["X"]  = maybe_flatten_X(b["test"]["X"],  flatten=True)
        bundles.append(b)

    # prints (opcional)
    for b in bundles:
        print(f"H{b['horizon']} Train:", b["train"]["X"].shape, b["train"]["y"].shape)
        print(f"H{b['horizon']} Valid:", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print(f"H{b['horizon']} Test :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

In [14]:
#bundle_delta_60, bundle_delta_90 = create_bundles(window_size = 30, targets = ['delta_60', 'delta_90'], windows_paths = windows_paths, scalers_paths = scalers_paths, flatten_X = False)
#bundle_ret_60, bundle_ret_90 = create_bundles(window_size = 30, targets = ['ret_60', 'ret_90'], windows_paths = windows_paths, scalers_paths = scalers_paths)
'''
def run_mlp(window_size: int, *, alpha: float = 1.0, verbose: bool = True):

    size = window_size

    if verbose:
        print("\n" + "=" * 80)
        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{size} | alpha={alpha}")
        print("=" * 80)

    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []

    for target in targets:
        if verbose:
            print(f"\n[BUILD] L{size} | target = '{target}'")


        # crear SOLO 1 bundle (y aplanar X para Ridge)
        (bundle,) = create_bundles(
            window_size=size,
            targets=[target],            # <- SOLO UNO
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,              # <- MLP necesita 2D
        )
'''


'\ndef run_mlp(window_size: int, *, alpha: float = 1.0, verbose: bool = True):\n\n    size = window_size\n\n    if verbose:\n        print("\n" + "=" * 80)\n        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{size} | alpha={alpha}")\n        print("=" * 80)\n\n    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]\n    rows = []\n\n    for target in targets:\n        if verbose:\n            print(f"\n[BUILD] L{size} | target = \'{target}\'")\n\n\n        # crear SOLO 1 bundle (y aplanar X para Ridge)\n        (bundle,) = create_bundles(\n            window_size=size,\n            targets=[target],            # <- SOLO UNO\n            windows_paths=windows_paths,\n            scalers_paths=scalers_paths,\n            flatten_X=True,              # <- MLP necesita 2D\n        )\n'

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [15]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [16]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [17]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [18]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Gestión de dataset de métricas**

In [19]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [20]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

## **9. Métricas ML**

In [21]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

In [22]:
def get_metrics_torch(bundle, model, *, device) -> tuple[dict, dict]:
  # -------- VALID --------
  X_valid = bundle["valid"]["X"]
  y_valid = bundle["valid"]["y"]
  y_pred_valid = predict_mlp(model, X_valid, device=device)
  metrics_valid = compute_seq2one_metrics(y_valid, y_pred_valid, compute_r2=True)

  # -------- TEST --------
  X_test = bundle["test"]["X"]
  y_test = bundle["test"]["y"]
  y_pred_test = predict_mlp(model, X_test, device=device)
  metrics_test  = compute_seq2one_metrics(y_test, y_pred_test,  compute_r2=True)

  return metrics_valid, metrics_test

# **DEFINICIÓN DE MODELO**

## **11. Definición del modelo — placeholder**

### **11.1. Baseline MLP Configuration (SEQ2ONE – delta_60)**


**Hiperparámetros baseline**


| Hiperparámetro   | Valor   |
|------------------|---------|
| in_dim           | 1200    |
| hidden_dim       | 128     |
| dropout          | 0.0     |
| learning_rate    | 1e-3    |
| weight_decay     | 1e-4    |
| max_epochs       | 30      |
| patience         | 5       |
| optimizer        | Adam    |
| loss_function    | MSE     |

In [23]:
import pandas as pd
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

df_mlp_all_sizes = load_seq2one_metrics_if_exists(name="mlp")
df_mlp_delta_60 = df_mlp_all_sizes[df_mlp_all_sizes["target"] == "delta_60"].copy()

In [24]:
df_mlp_delta_60

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,w_decay
0,mlp,test,30,delta_60,60,50.232999,77.094085,0.264331,0.651685,0.0001
1,mlp,valid,30,delta_60,60,32.880210,46.714091,0.273345,0.661028,0.0001
8,mlp,test,60,delta_60,60,38.923520,63.419347,0.496504,0.738393,0.0001
9,mlp,valid,60,delta_60,60,24.081825,37.120464,0.550355,0.758303,0.0001
16,mlp,test,90,delta_60,60,40.382358,64.906080,0.481965,0.729042,0.0001
17,mlp,valid,90,delta_60,60,25.324535,38.475513,0.528487,0.741210,0.0001
24,mlp,test,120,delta_60,60,39.748844,63.426119,0.518663,0.741398,0.0001
25,mlp,valid,120,delta_60,60,24.769221,37.699327,0.556814,0.754523,0.0001
32,mlp,test,180,delta_60,60,38.112160,60.078973,0.570888,0.783965,0.0001
33,mlp,valid,180,delta_60,60,23.693735,35.268546,0.635576,0.785406,0.0001


### **11.2. Imports (PyTorch) + semillas**

In [25]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [26]:
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

### **11.2. Dataset/DataLoader desde bundle**

In [27]:
#DATASET/DATALOADER DESDE BUNDLE

def make_loaders_from_bundle(
    bundle,
    *,
    batch_size: int = 4096,
    num_workers: int = 0,
) -> dict:
    """
    Convierte bundle {train/valid/test} a DataLoaders PyTorch.
    Espera X: (n, 1200) y y: (n,)
    """
    loaders = {}

    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)  # (n,1)

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )

    return loaders

In [28]:
#loaders_60 = make_loaders_from_bundle(bundle_60, batch_size=16384)
#loaders_90 = make_loaders_from_bundle(bundle_90, batch_size=16384)

### **11.3. Definición del modelo MLP (simple y controlado)**

In [29]:
#DEFINICIÓN DEL MODELO MLP
class MLPSeq2One(nn.Module):
    def __init__(self, in_dim: int = 1200, hidden_dim: int = 128, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

### **11.4. Entrenamiento con early stopping (VALID)**

In [30]:
#ENTRENAMIENTO CON EARLY STOPPING (VALID)
@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)

import time
import torch
import torch.nn as nn

def _ts():
    return time.strftime("%H:%M:%S")

def train_mlp(
    loaders: dict,
    *,
    in_dim: int = 1200,
    hidden_dim: int = 128,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    device: torch.device,
    verbose: bool = True,
    log_every: int = 0,  # 0 => no log por batch; si pones 50, log cada 50 batches
):
    """
    Entrena un MLP seq2one usando TRAIN y early stopping en VALID (por MSE).
    Logs: inicio/fin, train_loss, valid_mse, mejoras, early-stopping.
    """
    if verbose:
        print(f"[{_ts()}] [TRAIN] START | in_dim={in_dim} hidden_dim={hidden_dim} dropout={dropout} "
              f"lr={lr} wd={weight_decay} max_epochs={max_epochs} patience={patience} "
              f"device={device.type}")

    t_global = time.perf_counter()

    model = MLPSeq2One(in_dim=in_dim, hidden_dim=hidden_dim, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        t_epoch = time.perf_counter()

        # -------------------------
        # TRAIN EPOCH
        # -------------------------
        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for b, (xb, yb) in enumerate(loaders["train"], start=1):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            # acumular loss para log por época
            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

            if verbose and log_every and (b % log_every == 0):
                train_loss_avg_so_far = train_loss_sum / max(train_n, 1)
                print(f"[{_ts()}]   epoch={epoch:02d} batch={b:04d} | train_loss_avg={train_loss_avg_so_far:.6f}")

        train_loss_avg = train_loss_sum / max(train_n, 1)

        # -------------------------
        # VALIDATION
        # -------------------------
        #if verbose:
        #    print(f"[{_ts()}]   [VALID] Calculando valid_mse ...")
        t0 = time.perf_counter()

        valid_mse = evaluate_mse(model, loaders["valid"], device)

        dt_valid = time.perf_counter() - t0
        dt_epoch = time.perf_counter() - t_epoch

        # -------------------------
        # EARLY STOPPING LOGIC
        # -------------------------
        improved = valid_mse < (best_valid - 1e-9)
        if improved:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            print(
                f"[{_ts()}] epoch={epoch:02d} | train_loss={train_loss_avg:.6f} | "
                f"valid_mse={valid_mse:.6f} | {flag} | dt_valid={dt_valid:.2f}s | dt_epoch={dt_epoch:.2f}s"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"[{_ts()}] [TRAIN] EARLY STOPPING | patience={patience} | best_valid_mse={best_valid:.6f}")
            break

    if best_state is not None:
        if verbose:
            print(f"[{_ts()}] [TRAIN] Cargando best_state (best_valid_mse={best_valid:.6f}) ...")
        model.load_state_dict(best_state)

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [TRAIN] END | best_valid_mse={best_valid:.6f} | dt_total={dt_all:.2f}s")

    return model


### **11.5. Predicciones MLP**


In [31]:
#predicciones MLP

import numpy as np
import torch

@torch.no_grad()
def predict_mlp(model, X: np.ndarray, *, device: torch.device, batch_size: int = 32768) -> np.ndarray:
    """
    Predice con un modelo MLP PyTorch en batches.
    Retorna shape (n_samples,)
    """
    model.eval()

    X = np.asarray(X, dtype=np.float32)
    n = X.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device)
        yb = model(xb).squeeze(-1)          # (batch,)
        preds.append(yb.detach().cpu().numpy())

    return np.concatenate(preds, axis=0)


## **12. Ejecución completa**

In [32]:
#RUN MLP

import pandas as pd
import gc
import time

def _ts():
    return time.strftime("%H:%M:%S")

def run_mlp(window_size: int, *, weight_decay: float = 1e-4, verbose: bool = True):

    size = window_size
    n_features = 36

    if verbose:
        print("\n" + "=" * 80)
        print(f"[{_ts()}] MLP | SEQ2ONE | WINDOW_SIZE=L{size} | in_dim={size*n_features} | weight_decay={weight_decay}")
        print("=" * 80)

    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []

    t_global = time.perf_counter()

    for i, target in enumerate(targets, start=1):
        t_target = time.perf_counter()

        if verbose:
            print(f"\n[{_ts()}] [{i}/{len(targets)}] START target='{target}' | L{size}")

        # -------------------------
        # BUILD BUNDLE
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [BUILD] Creando bundle (flatten_X=True) ...")
        t0 = time.perf_counter()

        (bundle,) = create_bundles(
            window_size=size,
            targets=[target],
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,  # MLP necesita 2D
        )

        if verbose:
            dt = time.perf_counter() - t0
            # (opcional) si existe el shape, lo mostramos
            try:
                xshape = bundle["train"]["X"].shape
                yshape = bundle["train"]["y"].shape
                print(f"[{_ts()}]   [BUILD] OK | train X={xshape} y={yshape} | dt={dt:.2f}s")
            except Exception:
                print(f"[{_ts()}]   [BUILD] OK | dt={dt:.2f}s")

        # -------------------------
        # LOADERS
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [LOADERS] Creando DataLoaders ...")
        t0 = time.perf_counter()

        loaders = make_loaders_from_bundle(
            bundle,
            batch_size=16384
        )

        if verbose:
            dt = time.perf_counter() - t0
            # si los loaders exponen dataset length, lo imprimimos
            try:
                ntr = len(loaders["train"].dataset)
                nva = len(loaders["valid"].dataset)
                nte = len(loaders["test"].dataset)
                print(f"[{_ts()}]   [LOADERS] OK | n(train/valid/test)=({ntr}/{nva}/{nte}) | dt={dt:.2f}s")
            except Exception:
                print(f"[{_ts()}]   [LOADERS] OK | dt={dt:.2f}s")

        # -------------------------
        # TRAIN
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [TRAIN] Iniciando entrenamiento ...")
        t0 = time.perf_counter()

        model = train_mlp(
            loaders,
            in_dim=size * n_features,
            hidden_dim=128,
            dropout=0.0,
            lr=1e-3,
            weight_decay=weight_decay,
            max_epochs=30,
            patience=5,
            device=device
        )

        if verbose:
            dt = time.perf_counter() - t0
            print(f"[{_ts()}]   [TRAIN] FIN entrenamiento | dt={dt:.2f}s")

        # liberar TRAIN (opcional)
        if verbose:
            print(f"[{_ts()}]   [MEM] Liberando bundle['train'] y gc.collect() ...")
        del bundle["train"]
        gc.collect()

        # -------------------------
        # PRED + METRICS
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [PRED] Iniciando predicciones + cálculo de métricas (valid/test) ...")
        t0 = time.perf_counter()

        metrics_valid, metrics_test = get_metrics_torch(bundle, model, device=device)

        if verbose:
            dt = time.perf_counter() - t0
            print(f"[{_ts()}]   [METRICS] OK (valid/test) | dt={dt:.2f}s")

        # -------------------------
        # DF APPEND
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [DF] Agregando filas a la tabla de métricas ...")
        t0 = time.perf_counter()

        rows.append(metrics_to_df(
            metrics_valid,
            model="mlp",
            split="valid",
            horizon=bundle["horizon"],
            window_size=bundle["window_size"],
            target=bundle["target"],
        ))

        rows.append(metrics_to_df(
            metrics_test,
            model="mlp",
            split="test",
            horizon=bundle["horizon"],
            window_size=bundle["window_size"],
            target=bundle["target"],
        ))

        if verbose:
            dt = time.perf_counter() - t0
            print(f"[{_ts()}]   [DF] OK | dt={dt:.2f}s")

        # -------------------------
        # CLEANUP
        # -------------------------
        if verbose:
            print(f"[{_ts()}]   [CLEAN] Liberando objetos (bundle/model/metrics/loaders) ...")
        del bundle, model, metrics_valid, metrics_test, loaders
        gc.collect()

        if verbose:
            dt_target = time.perf_counter() - t_target
            print(f"[{_ts()}] [{i}/{len(targets)}] DONE target='{target}' | dt_total={dt_target:.2f}s")

    # -------------------------
    # FINAL DF
    # -------------------------
    if verbose:
        print(f"\n[{_ts()}] [FINAL] Concatenando resultados ...")
    t0 = time.perf_counter()

    df_mlp_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        dt = time.perf_counter() - t0
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [FINAL] OK | rows={len(df_mlp_metrics)} | dt_concat={dt:.2f}s | dt_total={dt_all:.2f}s")
        print(df_mlp_metrics[["window_size", "target", "split", "horizon_min", "model"]]
              .drop_duplicates()
              .to_string(index=False))

    return df_mlp_metrics


In [33]:
#RUN MLP INCREMENTAL

def run_mlp_incremental(
    window_sizes: list[int],
    *,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    name: str = "mlp",  # -> seq2one_ridge_metrics.parquet
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:

    # 1) Cargar si existe
    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    # 2) Asegurar columna w_decay
    if df_all.empty:
        df_all = pd.DataFrame(columns=[
            "model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA","w_decay"
        ])
    if "w_decay" not in df_all.columns:
        df_all["w_decay"] = pd.NA

    key_cols = ["model","w_decay","window_size","target","split","horizon_min"]

    # 3) Normalizar tipos (evita falsos mismatches)
    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")

    # 4) Loop por window_size
    for ws in window_sizes:

        # Si ya existen todas las filas esperadas para este ws, saltar
        # Esperadas: 4 targets x 2 splits (valid/test) = 8 filas para mlp (siempre)
        # (Esto asume que tu run_mlp devuelve valid+test para los 4 targets)
        df_ws = df_all[(df_all["model"] == "mlp") & (df_all["w_decay"] == weight_decay) & (df_all["window_size"] == ws)]
        if len(df_ws) >= 8:
            if verbose:
                print(f"[SKIP] L{ws}: ya hay {len(df_ws)} filas (mlp w_decay={weight_decay}).")
            continue

        if verbose:
            print("\n" + "="*90)
            print(f"[RUN] MLP incremental | L{ws} |in_dim={ws*36}| dropout={dropout} | lr={lr} | w_decay={weight_decay}")  ###################
            print("="*90)

        # 5) Entrenar y obtener métricas para ESTE ws
        df_new = run_mlp(ws, weight_decay=weight_decay, verbose=verbose).copy() #####################
        df_new["w_decay"] = weight_decay  # agregar

        # 6) Filtrar filas ya existentes (anti-duplicados)
        # Creamos un "key" para comparar rápido
        existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
        mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
        df_new = df_new.loc[mask_keep].copy()

        if df_new.empty:
            if verbose:
                print(f"[INFO] L{ws}: no había filas nuevas para agregar.")
            continue

        # 7) Merge + dedupe por seguridad
        df_all = pd.concat([df_all, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

        # 8) Guardar checkpoint con TU función
        save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

        if verbose:
            print(f"[OK] Checkpoint guardado. Total rows={len(df_all)}")

    return df_all

In [34]:
#df_mlp_all_sizes = load_seq2one_metrics_if_exists(name="mlp", base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics")

In [35]:
#MLP_ALL_TRAIN = '''
df_mlp_all_sizes = run_mlp_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="mlp",   # genera seq2one_ridge_metrics.parquet
    verbose=True,
)
#'''

[SKIP] L30: ya hay 8 filas (mlp w_decay=0.0001).
[SKIP] L60: ya hay 8 filas (mlp w_decay=0.0001).
[SKIP] L90: ya hay 8 filas (mlp w_decay=0.0001).
[SKIP] L120: ya hay 8 filas (mlp w_decay=0.0001).
[SKIP] L180: ya hay 8 filas (mlp w_decay=0.0001).


## **13. Tuning de Arquitectura (capacidad del modelo)**

Afecta el bias–variance trade-off estructural.

Parámetros relevantes:
- hidden_dim
- n_layers (si agregas más capas)
- dropout
- Activación (ReLU, GELU, etc.)

Esto controla qué tan complejo puede ser el mapeo no lineal entre la ventana y delta_60.

### **13.1. Funciones para tuning de arquitectura**

In [36]:
import torch
import torch.nn as nn

# ============================================================
# MLP configurable para tuning de arquitectura (SEQ2ONE)
# ============================================================
class MLPSeq2One(nn.Module):
    """
    MLP SEQ2ONE configurable.
    - in_dim: dimensión de entrada (L * F si está aplanado)
    - hidden_dims: lista con el tamaño de cada capa oculta (profundidad + anchura)
    - dropout: dropout aplicado entre capas (0.0 desactiva)
    - activation: "relu" o "gelu"
    """
    def __init__(
        self,
        *,
        in_dim: int = 1200,
        hidden_dims: list[int] = [128],
        dropout: float = 0.0,
        activation: str = "relu",
    ):
        super().__init__()

        if not hidden_dims or any(h <= 0 for h in hidden_dims):
            raise ValueError(f"hidden_dims inválido: {hidden_dims}")

        if activation.lower() == "relu":
            Act = nn.ReLU
        elif activation.lower() == "gelu":
            Act = nn.GELU
        else:
            raise ValueError(f"activation no soportada: {activation} (use 'relu' o 'gelu')")

        layers: list[nn.Module] = []
        prev = in_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(Act())
            if dropout and dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [37]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ============================================================
# EVAL: MSE (VALID) para early stopping
# ============================================================
@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)


def _ts():
    return time.strftime("%H:%M:%S")


# ============================================================
# TRAIN: MLP seq2one + early stopping en VALID (por MSE)
# ============================================================
def train_mlp(
    loaders: dict,
    *,
    # ---- arquitectura ----
    in_dim: int = 1200,
    hidden_dims: list[int] | None = None,   # ej: [256] o [256,128]
    dropout: float = 0.0,
    activation: str = "relu",               # "relu" o "gelu"

    # ---- entrenamiento ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    device: torch.device,
    verbose: bool = True,
    log_every: int = 0,  # 0 => no log por batch; si pones 50, log cada 50 batches
):
    """
    Entrena un MLP seq2one usando TRAIN y early stopping en VALID (por MSE).
    Logs: inicio/fin, train_loss, valid_mse, mejoras, early-stopping.

    Nota:
    - hidden_dims controla profundidad/anchura. Si None, usa [128] por compatibilidad.
    """
    if hidden_dims is None:
        hidden_dims = [128]  # baseline equivalente a hidden_dim=128

    if verbose:
        print(
            f"[{_ts()}] [TRAIN] START | in_dim={in_dim} hidden_dims={hidden_dims} "
            f"dropout={dropout} activation={activation} lr={lr} wd={weight_decay} "
            f"max_epochs={max_epochs} patience={patience} device={device.type}"
        )

    t_global = time.perf_counter()

    # ---- modelo (arquitectura tuneable) ----
    model = MLPSeq2One(
        in_dim=in_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
        activation=activation,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        t_epoch = time.perf_counter()

        # -------------------------
        # TRAIN EPOCH
        # -------------------------
        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for b, (xb, yb) in enumerate(loaders["train"], start=1):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

            if verbose and log_every and (b % log_every == 0):
                train_loss_avg_so_far = train_loss_sum / max(train_n, 1)
                print(f"[{_ts()}]   epoch={epoch:02d} batch={b:04d} | train_loss_avg={train_loss_avg_so_far:.6f}")

        train_loss_avg = train_loss_sum / max(train_n, 1)

        # -------------------------
        # VALIDATION (MSE)
        # -------------------------
        t0 = time.perf_counter()
        valid_mse = evaluate_mse(model, loaders["valid"], device)
        dt_valid = time.perf_counter() - t0
        dt_epoch = time.perf_counter() - t_epoch

        # -------------------------
        # EARLY STOPPING
        # -------------------------
        improved = valid_mse < (best_valid - 1e-9)
        if improved:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            print(
                f"[{_ts()}] epoch={epoch:02d} | train_loss={train_loss_avg:.6f} | "
                f"valid_mse={valid_mse:.6f} | {flag} | dt_valid={dt_valid:.2f}s | dt_epoch={dt_epoch:.2f}s"
            )

        if bad_epochs >= patience:
            if verbose:
                print(f"[{_ts()}] [TRAIN] EARLY STOPPING | patience={patience} | best_valid_mse={best_valid:.6f}")
            break

    if best_state is not None:
        if verbose:
            print(f"[{_ts()}] [TRAIN] Cargando best_state (best_valid_mse={best_valid:.6f}) ...")
        model.load_state_dict(best_state)

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [TRAIN] END | best_valid_mse={best_valid:.6f} | dt_total={dt_all:.2f}s")

    return model


In [38]:
#cfg = {"hidden_dims": [256, 128], "dropout": 0.1, "activation": "relu"}
#model = train_mlp(loaders, device=device, in_dim=1200, **cfg)

In [39]:
import numpy as np
import torch

@torch.no_grad()
def predict_mlp(
    model,
    X: np.ndarray,
    *,
    device: torch.device,
    batch_size: int = 32768
) -> np.ndarray:
    """
    Predice con un modelo MLP PyTorch en batches.
    Retorna shape (n_samples,)
    """
    model.eval()

    X = np.asarray(X, dtype=np.float32)
    n = X.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb_np = np.ascontiguousarray(X[i : i + batch_size])
        xb = torch.from_numpy(xb_np).to(device, non_blocking=True)

        yb = model(xb).squeeze(-1)  # (batch,)
        preds.append(yb.detach().cpu().numpy())

    return np.concatenate(preds, axis=0)


In [40]:
# ============================================================
# RUN MLP (solo delta_60 + arquitectura tuneable)
# ============================================================
import pandas as pd
import gc
import time

def _ts():
    return time.strftime("%H:%M:%S")


def run_mlp_delta_60(
    window_size: int,
    *,
    hidden_dims: list[int] = [128],
    dropout: float = 0.0,
    activation: str = "relu",
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    batch_size: int = 16384,
    verbose: bool = True,
) -> pd.DataFrame:

    size = window_size
    n_features = 36
    target = "delta_60"

    if verbose:
        print("\n" + "=" * 90)
        print(
            f"[{_ts()}] MLP | SEQ2ONE | target={target} | L{size} | in_dim={size*n_features} | "
            f"hidden_dims={hidden_dims} | dropout={dropout} | act={activation} | "
            f"lr={lr} | w_decay={weight_decay}"
        )
        print("=" * 90)

    rows = []
    t_global = time.perf_counter()

    # -------------------------
    # BUILD BUNDLE
    # -------------------------
    if verbose:
        print(f"[{_ts()}]   [BUILD] Creando bundle (flatten_X=True) ...")
    t0 = time.perf_counter()

    (bundle,) = create_bundles(
        window_size=size,
        targets=[target],
        windows_paths=windows_paths,
        scalers_paths=scalers_paths,
        flatten_X=True,  # MLP necesita 2D
    )

    if verbose:
        dt = time.perf_counter() - t0
        try:
            xshape = bundle["train"]["X"].shape
            yshape = bundle["train"]["y"].shape
            print(f"[{_ts()}]   [BUILD] OK | train X={xshape} y={yshape} | dt={dt:.2f}s")
        except Exception:
            print(f"[{_ts()}]   [BUILD] OK | dt={dt:.2f}s")

    # -------------------------
    # LOADERS
    # -------------------------
    if verbose:
        print(f"[{_ts()}]   [LOADERS] Creando DataLoaders ...")
    t0 = time.perf_counter()

    loaders = make_loaders_from_bundle(
        bundle,
        batch_size=batch_size,
    )

    if verbose:
        dt = time.perf_counter() - t0
        try:
            ntr = len(loaders["train"].dataset)
            nva = len(loaders["valid"].dataset)
            nte = len(loaders["test"].dataset)
            print(f"[{_ts()}]   [LOADERS] OK | n(train/valid/test)=({ntr}/{nva}/{nte}) | dt={dt:.2f}s")
        except Exception:
            print(f"[{_ts()}]   [LOADERS] OK | dt={dt:.2f}s")

    # -------------------------
    # TRAIN
    # -------------------------
    if verbose:
        print(f"[{_ts()}]   [TRAIN] Iniciando entrenamiento ...")
    t0 = time.perf_counter()

    model = train_mlp(
        loaders,
        in_dim=size * n_features,
        hidden_dims=hidden_dims,
        dropout=dropout,
        activation=activation,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=30,
        patience=5,
        device=device,
        verbose=verbose,
    )

    if verbose:
        dt = time.perf_counter() - t0
        print(f"[{_ts()}]   [TRAIN] FIN entrenamiento | dt={dt:.2f}s")

    # liberar TRAIN (opcional)
    if verbose:
        print(f"[{_ts()}]   [MEM] Liberando bundle['train'] y gc.collect() ...")
    del bundle["train"]
    gc.collect()

    # -------------------------
    # PRED + METRICS
    # -------------------------
    if verbose:
        print(f"[{_ts()}]   [PRED] Predicciones + métricas (valid/test) ...")
    t0 = time.perf_counter()

    metrics_valid, metrics_test = get_metrics_torch(bundle, model, device=device)

    if verbose:
        dt = time.perf_counter() - t0
        print(f"[{_ts()}]   [METRICS] OK (valid/test) | dt={dt:.2f}s")

    # -------------------------
    # DF APPEND
    # -------------------------
    rows.append(metrics_to_df(
        metrics_valid,
        model="mlp",
        split="valid",
        horizon=bundle["horizon"],
        window_size=bundle["window_size"],
        target=bundle["target"],
    ))

    rows.append(metrics_to_df(
        metrics_test,
        model="mlp",
        split="test",
        horizon=bundle["horizon"],
        window_size=bundle["window_size"],
        target=bundle["target"],
    ))

    df_mlp_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    # ---- agregar columnas de arquitectura + entrenamiento para tracking ----
    df_mlp_metrics["hidden_dims"] = str(hidden_dims)
    df_mlp_metrics["dropout"] = dropout
    df_mlp_metrics["activation"] = activation
    df_mlp_metrics["lr"] = lr
    df_mlp_metrics["w_decay"] = weight_decay

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [DONE] rows={len(df_mlp_metrics)} | dt_total={dt_all:.2f}s")

    # cleanup
    del bundle, model, metrics_valid, metrics_test, loaders
    gc.collect()

    return df_mlp_metrics


In [41]:
# ============================================================
# RUN MLP INCREMENTAL (solo delta_60 + arquitectura tuneable)
# ============================================================
import pandas as pd

def run_mlp_incremental_delta_60(
    window_sizes: list[int],
    *,
    hidden_dims: list[int] = [128],
    dropout: float = 0.0,
    activation: str = "relu",
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    name: str = "mlp_arch_tuning_delta_60",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:

    # 1) Cargar si existe
    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    # 2) Esquema mínimo esperado (ahora guardamos arquitectura)
    base_cols = [
        "model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA",
        "hidden_dims","dropout","activation","lr","w_decay"
    ]
    if df_all.empty:
        df_all = pd.DataFrame(columns=base_cols)
    else:
        for c in base_cols:
            if c not in df_all.columns:
                df_all[c] = pd.NA

    # 3) Normalizar tipos
    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")

    # 4) Keys únicas por config + ws + split/horizon
    hidden_dims_str = str(hidden_dims)
    key_cols = ["model","window_size","target","split","horizon_min","hidden_dims","dropout","activation","lr","w_decay"]

    # 5) Loop por window_size
    for ws in window_sizes:

        df_ws = df_all[
            (df_all["model"] == "mlp") &
            (df_all["window_size"] == ws) &
            (df_all["target"] == "delta_60") &
            (df_all["hidden_dims"] == hidden_dims_str) &
            (df_all["dropout"] == dropout) &
            (df_all["activation"] == activation) &
            (df_all["lr"] == lr) &
            (df_all["w_decay"] == weight_decay)
        ]

        # Esperadas: delta_60 x 2 splits (valid/test) = 2 filas
        if len(df_ws) >= 2:
            if verbose:
                print(f"[SKIP] L{ws}: ya hay {len(df_ws)} filas para esta config (delta_60).")
            continue

        if verbose:
            print("\n" + "="*100)
            print(
                f"[RUN] MLP incremental | target=delta_60 | L{ws} | in_dim={ws*36} | "
                f"hidden_dims={hidden_dims} | dropout={dropout} | act={activation} | "
                f"lr={lr} | w_decay={weight_decay}"
            )
            print("="*100)

        # Entrenar/evaluar para ESTE ws (solo delta_60)
        df_new = run_mlp_delta_60(
            ws,
            hidden_dims=hidden_dims,
            dropout=dropout,
            activation=activation,
            lr=lr,
            weight_decay=weight_decay,
            verbose=verbose,
        ).copy()

        # Anti-duplicados por key
        existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
        mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
        df_new = df_new.loc[mask_keep].copy()

        if df_new.empty:
            if verbose:
                print(f"[INFO] L{ws}: no había filas nuevas para agregar.")
            continue

        # Merge + dedupe
        df_all = pd.concat([df_all, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

        # Guardar checkpoint
        save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

        if verbose:
            print(f"[OK] Checkpoint guardado. Total rows={len(df_all)}")

    return df_all


### **13.2. Ejecución de tuning**

In [42]:
# ============================================================
# ARCH_GRID: solo arquitectura (tuning controlado)
# ============================================================
ARCH_GRID = [
    # 1 capa (baseline y variantes)
    {"hidden_dims": [64],  "dropout": 0.0, "activation": "relu"},
    {"hidden_dims": [128], "dropout": 0.0, "activation": "relu"},  # baseline
    {"hidden_dims": [256], "dropout": 0.0, "activation": "relu"},
    {"hidden_dims": [512], "dropout": 0.0, "activation": "relu"},

    # 1 capa + dropout
    {"hidden_dims": [128], "dropout": 0.1, "activation": "relu"},
    {"hidden_dims": [256], "dropout": 0.1, "activation": "relu"},
    {"hidden_dims": [256], "dropout": 0.2, "activation": "relu"},

    # 2 capas
    {"hidden_dims": [128, 64],   "dropout": 0.0, "activation": "relu"},
    {"hidden_dims": [256, 128],  "dropout": 0.0, "activation": "relu"},
    {"hidden_dims": [256, 128],  "dropout": 0.1, "activation": "relu"},

    # (opcional) activación alternativa
    {"hidden_dims": [256], "dropout": 0.1, "activation": "gelu"},
]


In [43]:
# ============================================================
# TUNING RUNNER: recorre ARCH_GRID y ejecuta incremental por config
# ============================================================
import time
import pandas as pd

def _ts():
    return time.strftime("%H:%M:%S")


def run_mlp_arch_tuning_delta_60(
    window_sizes: list[int],
    *,
    arch_grid: list[dict],
    lr: float = 1e-3,                 # fijo en tuning de arquitectura
    weight_decay: float = 1e-4,       # fijo en tuning de arquitectura
    base_name: str = "mlp_arch_tuning_delta_60",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Recorre un grid de arquitecturas y ejecuta:
      run_mlp_incremental_delta_60(window_sizes, hidden_dims, dropout, activation, lr, weight_decay)

    Guarda checkpoints incrementales por config (name único por arquitectura).
    Retorna el último df_all cargado/guardado (del último config).
    """
    last_df = pd.DataFrame()

    for k, cfg in enumerate(arch_grid, start=1):
        hidden_dims = cfg["hidden_dims"]
        dropout = float(cfg.get("dropout", 0.0))
        activation = cfg.get("activation", "relu")

        # name único por arquitectura (para no mezclar resultados)
        arch_tag = f"hd={'-'.join(map(str, hidden_dims))}_do={dropout}_act={activation}"
        name = f"{base_name}__{arch_tag}"

        if verbose:
            print("\n" + "=" * 110)
            print(f"[{_ts()}] [{k}/{len(arch_grid)}] ARCH TUNING | {arch_tag} | lr={lr} | w_decay={weight_decay}")
            print("=" * 110)

        last_df = run_mlp_incremental_delta_60(
            window_sizes,
            hidden_dims=hidden_dims,
            dropout=dropout,
            activation=activation,
            lr=lr,
            weight_decay=weight_decay,
            name=name,
            base_dir=base_dir,
            verbose=verbose,
        )

    return last_df


In [44]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [46]:
import pandas as pd

def run_mlp_incremental_delta_60_onefile(
    window_sizes: list[int],
    arch_grid: list[dict],
    *,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    name: str = "mlp_arch_tuning_delta_60_all",  # <-- UN SOLO DATASET
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:

    # 1) Cargar si existe
    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    # 2) Esquema esperado (incluye arquitectura)
    cols = [
        "model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA",
        "hidden_dims","dropout","activation","lr","w_decay"
    ]
    if df_all.empty:
        df_all = pd.DataFrame(columns=cols)
    else:
        for c in cols:
            if c not in df_all.columns:
                df_all[c] = pd.NA

    # 3) Normalizar tipos
    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")

    # 4) Key única por corrida (incluye hiperparams)
    key_cols = ["model","window_size","target","split","horizon_min","hidden_dims","dropout","activation","lr","w_decay"]

    # 5) Loop por arquitectura y window_size
    for k, cfg in enumerate(arch_grid, start=1):
        hidden_dims = cfg["hidden_dims"]
        dropout = float(cfg.get("dropout", 0.0))
        activation = cfg.get("activation", "relu")
        hidden_dims_str = str(hidden_dims)

        if verbose:
            print("\n" + "=" * 110)
            print(f"[{_ts()}] [{k}/{len(arch_grid)}] ARCH | hidden_dims={hidden_dims} | dropout={dropout} | act={activation}")
            print("=" * 110)

        for ws in window_sizes:

            # Esperadas: delta_60 x 2 splits (valid/test) = 2 filas
            df_ws = df_all[
                (df_all["model"] == "mlp") &
                (df_all["target"] == "delta_60") &
                (df_all["window_size"] == ws) &
                (df_all["hidden_dims"] == hidden_dims_str) &
                (df_all["dropout"] == dropout) &
                (df_all["activation"] == activation) &
                (df_all["lr"] == lr) &
                (df_all["w_decay"] == weight_decay)
            ]

            if len(df_ws) >= 2:
                if verbose:
                    print(f"[SKIP] L{ws}: ya existen valid+test para esta config.")
                continue

            if verbose:
                print("\n" + "-" * 90)
                print(f"[RUN] target=delta_60 | L{ws} | in_dim={ws*36} | hidden_dims={hidden_dims} | dropout={dropout} | act={activation}")
                print("-" * 90)

            # Entrenar/evaluar ESTE ws con ESTA arquitectura
            df_new = run_mlp_delta_60(
                ws,
                hidden_dims=hidden_dims,
                dropout=dropout,
                activation=activation,
                lr=lr,
                weight_decay=weight_decay,
                verbose=verbose,
            ).copy()

            # Anti-duplicados por key
            existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
            mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
            df_new = df_new.loc[mask_keep].copy()

            if df_new.empty:
                if verbose:
                    print(f"[INFO] L{ws}: no hubo filas nuevas para agregar.")
                continue

            # Merge + dedupe
            df_all = pd.concat([df_all, df_new], ignore_index=True)
            df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

            # Guardar SIEMPRE al mismo archivo
            save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

            if verbose:
                print(f"[OK] Guardado incremental (1 archivo). Total rows={len(df_all)}")

    return df_all


In [47]:
window_sizes = [60, 180]

df_all = run_mlp_incremental_delta_60_onefile(
    window_sizes,
    ARCH_GRID,
    lr=1e-3,
    weight_decay=1e-4,
    name="mlp_arch_tuning_delta_60_all",
    base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose=True,
)


[15:38:53] [1/11] ARCH | hidden_dims=[64] | dropout=0.0 | act=relu

------------------------------------------------------------------------------------------
[RUN] target=delta_60 | L60 | in_dim=2160 | hidden_dims=[64] | dropout=0.0 | act=relu
------------------------------------------------------------------------------------------

[15:38:53] MLP | SEQ2ONE | target=delta_60 | L60 | in_dim=2160 | hidden_dims=[64] | dropout=0.0 | act=relu | lr=0.001 | w_decay=0.0001
[15:38:53]   [BUILD] Creando bundle (flatten_X=True) ...
H60 Train: (330144, 2160) (330144,)
H60 Valid: (70590, 2160) (70590,)
H60 Test : (70952, 2160) (70952,)
Scaler H60: StandardScaler
[15:39:03]   [BUILD] OK | train X=(330144, 2160) y=(330144,) | dt=9.48s
[15:39:03]   [LOADERS] Creando DataLoaders ...
[15:39:03]   [LOADERS] OK | n(train/valid/test)=(330144/70590/70952) | dt=0.00s
[15:39:03]   [TRAIN] Iniciando entrenamiento ...
[15:39:03] [TRAIN] START | in_dim=2160 hidden_dims=[64] dropout=0.0 activation=relu lr=0.00

/tmp/ipython-input-723513373.py:95: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_all, df_new], ignore_index=True)


H60 Train: (220704, 6480) (220704,)
H60 Valid: (47190, 6480) (47190,)
H60 Test : (47432, 6480) (47432,)
Scaler H60: StandardScaler
[15:42:40]   [BUILD] OK | train X=(220704, 6480) y=(220704,) | dt=18.36s
[15:42:40]   [LOADERS] Creando DataLoaders ...
[15:42:40]   [LOADERS] OK | n(train/valid/test)=(220704/47190/47432) | dt=0.00s
[15:42:40]   [TRAIN] Iniciando entrenamiento ...
[15:42:40] [TRAIN] START | in_dim=6480 hidden_dims=[64] dropout=0.0 activation=relu lr=0.001 wd=0.0001 max_epochs=30 patience=5 device=cuda
[15:42:47] epoch=01 | train_loss=3521.184638 | valid_mse=2801.498623 | BEST | dt_valid=1.27s | dt_epoch=7.36s
[15:42:54] epoch=02 | train_loss=2847.599943 | valid_mse=2281.132359 | BEST | dt_valid=1.27s | dt_epoch=7.37s
[15:43:02] epoch=03 | train_loss=2515.380330 | valid_mse=2132.272897 | BEST | dt_valid=1.28s | dt_epoch=7.51s
[15:43:09] epoch=04 | train_loss=2323.981159 | valid_mse=2020.758084 | BEST | dt_valid=1.46s | dt_epoch=7.50s
[15:43:17] epoch=05 | train_loss=2211.01

### **13.3. Análisis de resultados**

In [49]:
df_mlp_delta_60

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,w_decay
0,mlp,test,30,delta_60,60,50.232999,77.094085,0.264331,0.651685,0.0001
1,mlp,valid,30,delta_60,60,32.880210,46.714091,0.273345,0.661028,0.0001
8,mlp,test,60,delta_60,60,38.923520,63.419347,0.496504,0.738393,0.0001
9,mlp,valid,60,delta_60,60,24.081825,37.120464,0.550355,0.758303,0.0001
16,mlp,test,90,delta_60,60,40.382358,64.906080,0.481965,0.729042,0.0001
17,mlp,valid,90,delta_60,60,25.324535,38.475513,0.528487,0.741210,0.0001
24,mlp,test,120,delta_60,60,39.748844,63.426119,0.518663,0.741398,0.0001
25,mlp,valid,120,delta_60,60,24.769221,37.699327,0.556814,0.754523,0.0001
32,mlp,test,180,delta_60,60,38.112160,60.078973,0.570888,0.783965,0.0001
33,mlp,valid,180,delta_60,60,23.693735,35.268546,0.635576,0.785406,0.0001


In [51]:
df_mlp_delta_60_valid = df_mlp_delta_60[df_mlp_delta_60["split"] == "valid"].copy()
df_mlp_delta_60_valid

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,w_decay
1,mlp,valid,30,delta_60,60,32.880210,46.714091,0.273345,0.661028,0.0001
9,mlp,valid,60,delta_60,60,24.081825,37.120464,0.550355,0.758303,0.0001
17,mlp,valid,90,delta_60,60,25.324535,38.475513,0.528487,0.741210,0.0001
25,mlp,valid,120,delta_60,60,24.769221,37.699327,0.556814,0.754523,0.0001
33,mlp,valid,180,delta_60,60,23.693735,35.268546,0.635576,0.785406,0.0001


In [48]:
df_mlp_tuning_arch_delta_60 = df_all.copy()
df_mlp_tuning_arch_delta_60

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,hidden_dims,dropout,activation,lr,w_decay
0,mlp,test,60,delta_60,60,40.068522,64.459744,0.479849,0.734468,[64],0.0,relu,0.001,0.0001
1,mlp,valid,60,delta_60,60,24.927824,37.995405,0.528908,0.751019,[64],0.0,relu,0.001,0.0001
2,mlp,test,180,delta_60,60,40.178311,62.111102,0.541368,0.775434,[64],0.0,relu,0.001,0.0001
3,mlp,valid,180,delta_60,60,25.115257,36.565634,0.608278,0.777230,[64],0.0,relu,0.001,0.0001
4,mlp,test,60,delta_60,60,38.833490,63.446381,0.496075,0.746188,[128],0.0,relu,0.001,0.0001
5,mlp,valid,60,delta_60,60,24.209347,37.239405,0.547468,0.757337,[128],0.0,relu,0.001,0.0001
6,mlp,test,180,delta_60,60,38.316973,59.955042,0.572657,0.783479,[128],0.0,relu,0.001,0.0001
7,mlp,valid,180,delta_60,60,23.858791,35.293936,0.635051,0.786341,[128],0.0,relu,0.001,0.0001
8,mlp,test,60,delta_60,60,37.773079,62.546829,0.510263,0.752019,[256],0.0,relu,0.001,0.0001
9,mlp,valid,60,delta_60,60,23.512603,36.644319,0.561816,0.766311,[256],0.0,relu,0.001,0.0001


In [52]:
df_mlp_tuning_arch_delta_60_valid  = df_mlp_tuning_arch_delta_60 [df_mlp_tuning_arch_delta_60 ["split"] == "valid"].copy()
df_mlp_tuning_arch_delta_60_valid

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,hidden_dims,dropout,activation,lr,w_decay
1,mlp,valid,60,delta_60,60,24.927824,37.995405,0.528908,0.751019,[64],0.0,relu,0.001,0.0001
3,mlp,valid,180,delta_60,60,25.115257,36.565634,0.608278,0.777230,[64],0.0,relu,0.001,0.0001
5,mlp,valid,60,delta_60,60,24.209347,37.239405,0.547468,0.757337,[128],0.0,relu,0.001,0.0001
7,mlp,valid,180,delta_60,60,23.858791,35.293936,0.635051,0.786341,[128],0.0,relu,0.001,0.0001
9,mlp,valid,60,delta_60,60,23.512603,36.644319,0.561816,0.766311,[256],0.0,relu,0.001,0.0001
11,mlp,valid,180,delta_60,60,23.066683,34.578408,0.649699,0.791799,[256],0.0,relu,0.001,0.0001
13,mlp,valid,60,delta_60,60,23.648664,36.740375,0.559516,0.764764,[512],0.0,relu,0.001,0.0001
15,mlp,valid,180,delta_60,60,23.369585,34.678097,0.647676,0.790461,[512],0.0,relu,0.001,0.0001
17,mlp,valid,60,delta_60,60,23.807150,36.866043,0.556497,0.759439,[128],0.1,relu,0.001,0.0001
19,mlp,valid,180,delta_60,60,23.068531,34.573680,0.649794,0.790843,[128],0.1,relu,0.001,0.0001


In [54]:
metrics = ["MAE", "RMSE", "R2", "DA"]

summary_valid = pd.DataFrame({
    "min": df_mlp_tuning_arch_delta_60_valid[metrics].min(),
    "max": df_mlp_tuning_arch_delta_60_valid[metrics].max(),
    "dif": df_mlp_tuning_arch_delta_60_valid[metrics].max() - df_mlp_tuning_arch_delta_60_valid[metrics].min(),
})

summary_valid

,min,max,dif
MAE,22.505675,25.115257,2.609582
RMSE,33.985249,37.995405,4.010157
R2,0.528908,0.661614,0.132706
DA,0.751019,0.796768,0.045749


In [56]:
df_mlp_tuning_arch_delta_60_test  = df_mlp_tuning_arch_delta_60 [df_mlp_tuning_arch_delta_60 ["split"] == "test"].copy()
df_mlp_tuning_arch_delta_60_test

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,hidden_dims,dropout,activation,lr,w_decay
0,mlp,test,60,delta_60,60,40.068522,64.459744,0.479849,0.734468,[64],0.0,relu,0.001,0.0001
2,mlp,test,180,delta_60,60,40.178311,62.111102,0.541368,0.775434,[64],0.0,relu,0.001,0.0001
4,mlp,test,60,delta_60,60,38.833490,63.446381,0.496075,0.746188,[128],0.0,relu,0.001,0.0001
6,mlp,test,180,delta_60,60,38.316973,59.955042,0.572657,0.783479,[128],0.0,relu,0.001,0.0001
8,mlp,test,60,delta_60,60,37.773079,62.546829,0.510263,0.752019,[256],0.0,relu,0.001,0.0001
10,mlp,test,180,delta_60,60,37.475574,59.351822,0.581213,0.782297,[256],0.0,relu,0.001,0.0001
12,mlp,test,60,delta_60,60,38.659862,63.227579,0.499545,0.742163,[512],0.0,relu,0.001,0.0001
14,mlp,test,180,delta_60,60,38.206245,59.290018,0.582084,0.775138,[512],0.0,relu,0.001,0.0001
16,mlp,test,60,delta_60,60,38.543919,63.874819,0.489246,0.738577,[128],0.1,relu,0.001,0.0001
18,mlp,test,180,delta_60,60,36.966631,58.563898,0.592258,0.791693,[128],0.1,relu,0.001,0.0001


In [57]:
summary_test = pd.DataFrame({
    "min": df_mlp_tuning_arch_delta_60_test[metrics].min(),
    "max": df_mlp_tuning_arch_delta_60_test[metrics].max(),
    "dif": df_mlp_tuning_arch_delta_60_test[metrics].max() - df_mlp_tuning_arch_delta_60_test[metrics].min(),
})

summary_test

,min,max,dif
MAE,36.232697,40.466921,4.234224
RMSE,58.366176,64.459744,6.093568
R2,0.479849,0.595007,0.115157
DA,0.718921,0.793382,0.074461


### **13.4. Conclusiones – Selección y congelamiento de arquitectura (MLP SEQ2ONE – delta_60)**



1. Evaluación exclusivamente en VALID  
   La selección de arquitectura se realizó únicamente utilizando métricas del split de validación, manteniendo el conjunto TEST completamente fuera del proceso de decisión.

2. Sensibilidad limitada a la arquitectura  
   Las diferencias entre configuraciones no son estructuralmente grandes.  
   - Rango total de MAE ≈ 2.61 puntos.  
   - Entre las mejores configuraciones (top 3), la diferencia es < 0.6 puntos.  
   Esto indica que el problema no es altamente sensible a incrementos agresivos de capacidad.

3. Ventana óptima  
   La ventana L=180 domina consistentemente a L=60 en todas las métricas (MAE, RMSE, R² y DA).  
   Se confirma que un contexto largo mejora la capacidad predictiva para delta_60.

4. Tamaño de capa oculta  
   - hidden_dims=[256] muestra desempeño consistentemente superior.
   - Incrementar a 512 no aporta mejoras.
   - Arquitecturas de dos capas no superan de forma clara a una sola capa de 256.

5. Efecto del dropout  
   - Dropout moderado (0.1–0.2) mejora ligeramente la generalización.
   - La mejor métrica de validación (MAE y RMSE mínimos, R² máximo, DA máximo) se obtiene con:
     
     hidden_dims = [256]  
     dropout = 0.2  
     activation = relu  

6. Decisión final de arquitectura (congelada)

   - window_size = 180  
   - hidden_dims = [256]  
   - dropout = 0.2  
   - activation = relu  
   - lr = 1e-3 (provisional)  
   - weight_decay = 1e-4 (provisional)

A partir de este punto, la arquitectura se considera fija.  
La siguiente etapa será el tuning de hiperparámetros de entrenamiento (learning rate y regularización), manteniendo constante la estructura del modelo.


## **14. Tuning de Aprendizaje**

### **14.1. Funciones**


In [70]:
import torch
import torch.nn as nn

class MLPSeq2One(nn.Module):
    """
    MLP SEQ2ONE (arquitectura congelada para delta_60).
    """
    def __init__(
        self,
        *,
        in_dim: int,
        hidden_dims: list[int] = [256],
        dropout: float = 0.2,
        activation: str = "relu",
    ):
        super().__init__()

        if activation.lower() == "relu":
            Act = nn.ReLU
        elif activation.lower() == "gelu":
            Act = nn.GELU
        else:
            raise ValueError(f"activation no soportada: {activation}")

        layers: list[nn.Module] = []
        prev = in_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(Act())
            if dropout and dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


In [71]:
HP_GRID_SMALL = [
    {"lr": 5e-4, "weight_decay": 1e-5},
    {"lr": 5e-4, "weight_decay": 1e-4},
    {"lr": 1e-3, "weight_decay": 1e-5},
    {"lr": 1e-3, "weight_decay": 1e-4},   # baseline
    {"lr": 2e-3, "weight_decay": 1e-4},
]
HP_GRID_SMALL


[{'lr': 0.0005, 'weight_decay': 1e-05},
 {'lr': 0.0005, 'weight_decay': 0.0001},
 {'lr': 0.001, 'weight_decay': 1e-05},
 {'lr': 0.001, 'weight_decay': 0.0001},
 {'lr': 0.002, 'weight_decay': 0.0001}]

1) Congelar arquitectura en train_mlp

In [72]:
# ============================================================
# Arquitectura congelada (delta_60, SEQ2ONE)
# ============================================================
FROZEN_ARCH = dict(
    hidden_dims=[256],
    dropout=0.2,
    activation="relu",
)


2) Runner para una corrida (L=180, delta_60) variando lr/wd

In [73]:
def run_mlp_lrwd_delta_60(
    *,
    window_size: int = 180,
    lr: float,
    weight_decay: float,
    verbose: bool = True,
) -> pd.DataFrame:
    return run_mlp_delta_60(
        window_size,
        hidden_dims=FROZEN_ARCH["hidden_dims"],
        dropout=FROZEN_ARCH["dropout"],
        activation=FROZEN_ARCH["activation"],
        lr=lr,
        weight_decay=weight_decay,
        verbose=verbose,
    )


3) Incremental “onefile” para guardar todo en un solo dataset

In [74]:
import pandas as pd
import time

def _ts():
    return time.strftime("%H:%M:%S")


def run_mlp_hp_tuning_delta_60_onefile(
    hp_grid: list[dict],
    *,
    window_size: int = 180,
    name: str = "mlp_learning_tuning_delta_60_L180",

    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:

    # 1) Cargar si existe
    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    # 2) Columnas esperadas (incluye hp y arch para trazabilidad)
    cols = [
        "model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA",
        "hidden_dims","dropout","activation","lr","w_decay"
    ]
    if df_all.empty:
        df_all = pd.DataFrame(columns=cols)
    else:
        for c in cols:
            if c not in df_all.columns:
                df_all[c] = pd.NA

    # 3) Normalizar tipos
    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")

    # 4) Key única por corrida
    key_cols = ["model","window_size","target","split","horizon_min","hidden_dims","dropout","activation","lr","w_decay"]

    # 5) Loop hiperparámetros
    for i, hp in enumerate(hp_grid, start=1):
        lr = float(hp["lr"])
        wd = float(hp["weight_decay"])

        # ¿ya existen valid+test para esta config?
        hidden_dims_str = str(FROZEN_ARCH["hidden_dims"])
        df_cfg = df_all[
            (df_all["model"] == "mlp") &
            (df_all["target"] == "delta_60") &
            (df_all["window_size"] == window_size) &
            (df_all["hidden_dims"] == hidden_dims_str) &
            (df_all["dropout"] == FROZEN_ARCH["dropout"]) &
            (df_all["activation"] == FROZEN_ARCH["activation"]) &
            (df_all["lr"] == lr) &
            (df_all["w_decay"] == wd)
        ]

        if len(df_cfg) >= 2:
            if verbose:
                print(f"[SKIP] [{i}/{len(hp_grid)}] lr={lr} wd={wd}: ya existe valid+test.")
            continue

        if verbose:
            print("\n" + "=" * 110)
            print(f"[{_ts()}] [RUN {i}/{len(hp_grid)}] HP TUNING | L{window_size} | lr={lr} | w_decay={wd} | arch={FROZEN_ARCH}")
            print("=" * 110)

        df_new = run_mlp_lrwd_delta_60(
            window_size=window_size,
            lr=lr,
            weight_decay=wd,
            verbose=verbose,
        ).copy()

        # Anti-duplicados por key
        existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
        mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
        df_new = df_new.loc[mask_keep].copy()

        if df_new.empty:
            if verbose:
                print(f"[INFO] lr={lr} wd={wd}: no hubo filas nuevas.")
            continue

        df_all = pd.concat([df_all, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

        save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

        if verbose:
            print(f"[OK] Guardado incremental. Total rows={len(df_all)}")

    return df_all


### **14.2. Ejecución con tu HP_GRID_SMALL**

In [75]:
df_hp = run_mlp_hp_tuning_delta_60_onefile(
    HP_GRID_SMALL,
    window_size=180,
    name="mlp_learning_tuning_delta_60_L180",
    base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose=True,
)


[17:30:09] [RUN 1/5] HP TUNING | L180 | lr=0.0005 | w_decay=1e-05 | arch={'hidden_dims': [256], 'dropout': 0.2, 'activation': 'relu'}

[17:30:09] MLP | SEQ2ONE | target=delta_60 | L180 | in_dim=6480 | hidden_dims=[256] | dropout=0.2 | act=relu | lr=0.0005 | w_decay=1e-05
[17:30:09]   [BUILD] Creando bundle (flatten_X=True) ...
H60 Train: (220704, 6480) (220704,)
H60 Valid: (47190, 6480) (47190,)
H60 Test : (47432, 6480) (47432,)
Scaler H60: StandardScaler
[17:30:28]   [BUILD] OK | train X=(220704, 6480) y=(220704,) | dt=18.27s
[17:30:28]   [LOADERS] Creando DataLoaders ...
[17:30:28]   [LOADERS] OK | n(train/valid/test)=(220704/47190/47432) | dt=0.00s
[17:30:28]   [TRAIN] Iniciando entrenamiento ...
[17:30:28] [TRAIN] START | in_dim=6480 hidden_dims=[256] dropout=0.2 activation=relu lr=0.0005 wd=1e-05 max_epochs=30 patience=5 device=cuda
[17:30:35] epoch=01 | train_loss=3501.621376 | valid_mse=2841.600890 | BEST | dt_valid=1.29s | dt_epoch=7.36s
[17:30:43] epoch=02 | train_loss=2899.5

/tmp/ipython-input-2990061145.py:86: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_all, df_new], ignore_index=True)


H60 Train: (220704, 6480) (220704,)
H60 Valid: (47190, 6480) (47190,)
H60 Test : (47432, 6480) (47432,)
Scaler H60: StandardScaler
[17:34:33]   [BUILD] OK | train X=(220704, 6480) y=(220704,) | dt=18.39s
[17:34:33]   [LOADERS] Creando DataLoaders ...
[17:34:33]   [LOADERS] OK | n(train/valid/test)=(220704/47190/47432) | dt=0.00s
[17:34:33]   [TRAIN] Iniciando entrenamiento ...
[17:34:33] [TRAIN] START | in_dim=6480 hidden_dims=[256] dropout=0.2 activation=relu lr=0.0005 wd=0.0001 max_epochs=30 patience=5 device=cuda
[17:34:40] epoch=01 | train_loss=3489.284654 | valid_mse=2826.119941 | BEST | dt_valid=1.29s | dt_epoch=7.48s
[17:34:48] epoch=02 | train_loss=2865.674952 | valid_mse=2296.862556 | BEST | dt_valid=1.27s | dt_epoch=7.54s
[17:34:55] epoch=03 | train_loss=2544.836942 | valid_mse=2135.147023 | BEST | dt_valid=1.30s | dt_epoch=7.52s
[17:35:03] epoch=04 | train_loss=2345.105408 | valid_mse=2009.037338 | BEST | dt_valid=1.30s | dt_epoch=7.54s
[17:35:10] epoch=05 | train_loss=2214.

In [76]:
df_hp

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,hidden_dims,dropout,activation,lr,w_decay
0,mlp,test,180,delta_60,60,38.083891,60.026711,0.571634,0.784830,[256],0.2,relu,0.0005,0.00001
1,mlp,valid,180,delta_60,60,24.080995,35.637654,0.627908,0.783325,[256],0.2,relu,0.0005,0.00001
2,mlp,test,180,delta_60,60,38.270353,60.490109,0.564995,0.786203,[256],0.2,relu,0.0005,0.00010
3,mlp,valid,180,delta_60,60,24.023231,35.728128,0.626017,0.784366,[256],0.2,relu,0.0005,0.00010
4,mlp,test,180,delta_60,60,36.319312,57.737049,0.603690,0.792094,[256],0.2,relu,0.0010,0.00001
5,mlp,valid,180,delta_60,60,22.961283,33.864160,0.664021,0.788762,[256],0.2,relu,0.0010,0.00001
6,mlp,test,180,delta_60,60,36.682615,58.471111,0.593549,0.790954,[256],0.2,relu,0.0010,0.00010
7,mlp,valid,180,delta_60,60,22.605512,34.016163,0.660998,0.796279,[256],0.2,relu,0.0010,0.00010
8,mlp,test,180,delta_60,60,40.108800,61.694373,0.547502,0.759977,[256],0.2,relu,0.0020,0.00010
9,mlp,valid,180,delta_60,60,23.635401,35.053303,0.640011,0.781669,[256],0.2,relu,0.0020,0.00010


### **14.3. Resultados**

In [77]:
df_hp_valid = df_hp[df_hp["split"] == "valid"].copy()
df_hp_valid

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,hidden_dims,dropout,activation,lr,w_decay
1,mlp,valid,180,delta_60,60,24.080995,35.637654,0.627908,0.783325,[256],0.2,relu,0.0005,0.00001
3,mlp,valid,180,delta_60,60,24.023231,35.728128,0.626017,0.784366,[256],0.2,relu,0.0005,0.00010
5,mlp,valid,180,delta_60,60,22.961283,33.864160,0.664021,0.788762,[256],0.2,relu,0.0010,0.00001
7,mlp,valid,180,delta_60,60,22.605512,34.016163,0.660998,0.796279,[256],0.2,relu,0.0010,0.00010
9,mlp,valid,180,delta_60,60,23.635401,35.053303,0.640011,0.781669,[256],0.2,relu,0.0020,0.00010


## Tuning de Hiperparámetros de Aprendizaje  
### MLP SEQ2ONE – delta_60 (Selección en VALID)

En esta etapa se mantuvo fija la arquitectura previamente seleccionada:

- window_size = 180  
- hidden_dims = [256]  
- dropout = 0.2  
- activation = relu  

Se variaron únicamente los hiperparámetros de aprendizaje:

- learning rate (lr)  
- weight decay (regularización L2)  

La selección se realizó exclusivamente utilizando el conjunto **VALID**.

---

## 1. Comparación de configuraciones (VALID)

| lr     | w_decay | MAE     | RMSE    | R²      | DA      |
|--------|---------|---------|---------|---------|---------|
| 5e-4   | 1e-5    | 24.0810 | 35.6377 | 0.6279  | 0.7833  |
| 5e-4   | 1e-4    | 24.0232 | 35.7281 | 0.6260  | 0.7844  |
| 1e-3   | 1e-5    | 22.9613 | **33.8642** | **0.6640** | 0.7888  |
| 1e-3   | 1e-4    | **22.6055** | 34.0162 | 0.6610  | **0.7963** |
| 2e-3   | 1e-4    | 23.6354 | 35.0533 | 0.6400  | 0.7817  |

---

## 2. Interpretación técnica

### 2.1 Learning rate

- lr = 5e-4 → desempeño inferior (MAE elevado).
- lr = 1e-3 → mejor desempeño global.
- lr = 2e-3 → degradación del rendimiento.

**Conclusión:**  
El rango óptimo de aprendizaje se encuentra en torno a **1e-3**.

---

### 2.2 Weight decay

Para lr = 1e-3:

- w_decay = 1e-5 → mejor RMSE y mejor R².
- w_decay = 1e-4 → mejor MAE y mejor Directional Accuracy.

Diferencia en MAE:

22.9613 vs 22.6055
↓
Mejora ≈ 0.36 puntos


Aunque la diferencia es moderada, es consistente.

---

## 3. Criterio de selección

Dado que la métrica primaria del proyecto es **MAE (error absoluto en puntos)**, la configuración seleccionada es:

lr = 1e-3
weight_decay = 1e-4


Porque:

- Presenta el MAE más bajo en VALID.
- Obtiene la mayor Directional Accuracy.
- Mantiene un R² prácticamente equivalente al máximo observado.

---

## 4. Configuración final congelada

### Arquitectura
- window_size = 180  
- hidden_dims = [256]  
- dropout = 0.2  
- activation = relu  

### Hiperparámetros de entrenamiento
- learning_rate = 1e-3  
- weight_decay = 1e-4  

A partir de este punto, tanto arquitectura como hiperparámetros quedan congelados.

---

## 5. Observación relevante

El baseline original (`lr=1e-3`, `weight_decay=1e-4`) resultó prácticamente óptimo tras el tuning.

Esto sugiere:

- Estabilidad del proceso de entrenamiento.
- Baja sensibilidad extrema a ajustes finos.
- Robustez estructural del modelo seleccionado.

---

## Próximo paso

Evaluación final en TEST con la configuración completamente congelada y comparación frente a modelos alternativos previamente entrenados.
